# 77. 漏斗图（px.funnel）

<!-- module-learning-arc:start -->
> **Plotly 模块主线｜第 14 / 18 步：表达层级、流程、贡献与地域**
>
> **持续应用背景：** 准备周度经营预警会：让读者通过悬停、缩放、下钻和层级探索，沿着异常、定位、行动的路径完成追问。
>
> **承接上一阶段：** 旭日图（px.sunburst）  →  **本章任务：** 漏斗图（px.funnel）  →  **下一步：** 瀑布图（Waterfall）
>
> **大作业连接：** 本章练习将成为《周度经营预警会：交互诊断与行动看板》的一部分，最终需要把诊断和行动视图组织成支持经营预警决策的可分享 HTML。
<!-- module-learning-arc:end -->


## 本章场景

当一整个业务流程被拆成清晰的前后步骤（从用户首次进入，到最终完成支付），每一步有多少人留下来、在哪一环流失最多，往往是运营最关心的问题。



## 本章目标

学完本章，你将能够：

- **理解**：理解「漏斗图（px.funnel）」的适用场景、数据结构要求，以及它想帮你读出的规律。
- **操作**：能按参数用相应绘图接口画出「漏斗图（px.funnel）」，并做必要的美化、注释与导出。
- **迁移**：能换一份真实经营数据，独立画出同类型的「漏斗图（px.funnel）」并读出其中的结论。


## 77.1 适用场景

**背景引入**：当一整个业务流程被拆成清晰的前后步骤（从用户首次进入，到最终完成支付），每一步有多少人留下来、在哪一环流失最多，往往是运营最关心的问题。之前学的柱状图、折线图更适合比较并列的类别，却很难把这种逐层递减的「漏斗」关系一次画清楚。漏斗图用每段宽度代表该阶段的人数，宽到窄的落差恰好对应每一步流失了多少用户，让你能一眼锁定转化率最低、最该改进的环节。（好比一场“筛人”的漏斗：最上面一步进来的人最多，每往下走一步就筛掉一批，漏斗越往下越窄，宽度就是剩下的人数；哪一段突然收窄，说明那段流失最狠、最该改。）

业务流程具有明确先后阶段，需要查看流失。


## 77.2 数据结构

有序阶段列和人数或数量列。


## 77.3 本章练习任务

运行基础图表后，完成以下任务：

1. 修改 textinfo 从 "value+percent initial+percent previous" 为 "value+percent previous"，对比整体转化率与相邻阶段转化率
2. 在分渠道漏斗中添加 hovertemplate 自定义悬停信息格式，查看各阶段精确人数
3. 将 color 映射为转化率指标，说明颜色编码对定位最大流失环节的作用


## 77.4 图表与参数速查

先用这张表建立本章的方法地图；每一行后面都有对应的独立示例或练习。

| 类别 | 常用方法或写法 | 主要用途 | 需要特别注意 |
| --- | --- | --- | --- |
| 基础图表 | `px.funnel()`、`fig.update_traces()`、`fig.update_layout()`、`fig.show()` | 业务流程具有明确先后阶段，需要查看流失。 | 阶段不是同一批用户 |
| 进阶变体 | `pd.DataFrame()`、`px.funnel()`、`fig.update_layout()`、`fig.show()` | 在基础图表上增加分组、注释、布局或交互 | 忽略时间窗口 |
| 关键参数 | `orientation` | 方向 | 阶段不是同一批用户 |
| 关键参数 | `textinfo` | 标签 | 忽略时间窗口 |
| 关键参数 | `funnelmode` | 分组 | 只看绝对流失不看转化率 |
| 关键参数 | `category_orders` | 阶段顺序 | 阶段不是同一批用户 |


## 77.5 准备可复现数据

先完成导入和数据准备，后续单元格只负责一种图表或一种分析动作。


<!-- math-foundation:chapter-77 -->
### 数学推导｜漏斗转化率

> 阅读方法：先跟着步骤理解每个量怎样产生，再看最后的可计算形式；不需要脱离业务场景死记公式。

**第 1 步｜相邻阶段比较。** 从阶段 $k-1$ 到 $k$ 的转化率是 $c_k=N_k/N_{k-1}$。

**第 2 步｜逐阶段连乘。** 从起点到阶段 $k$ 的累计转化为

$$
C_k=c_1c_2\cdots c_k
$$

**第 3 步｜中间人数会约掉。** 展开连乘后

$$
C_k=\frac{N_1}{N_0}\frac{N_2}{N_1}\cdots\frac{N_k}{N_{k-1}}=\frac{N_k}{N_0}
$$

这要求每一阶段人群都是前一阶段的子集。

**把上面的关系收束为本章计算式：**

$$
c_k=\frac{N_k}{N_{k-1}},\qquad C_k=\frac{N_k}{N_0}
$$

**符号解释：** $c_k$ 是相邻阶段转化率，$C_k$ 是从起点到阶段 $k$ 的累计转化率。

**代码对应：** 先确认阶段顺序与观察单位，再同时计算人数、相邻转化率和累计转化率。

**使用边界：** 阶段不是严格包含关系时不能使用漏斗；不同批次或时间窗也不能混合。


In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 1️⃣ 数据导入：手建两个小表 + 读取三个公开数据集
funnel = pd.DataFrame(
    {
        "stage": ["访问", "查看商品", "加入购物车", "提交订单", "支付成功"],
        "users": [12000, 7200, 3100, 1850, 1420],
    }
)
timeline = pd.DataFrame(
    {
        "task": ["数据准备", "探索分析", "图表制作", "报告复核"],
        "start": pd.to_datetime(
            ["2026-03-01", "2026-03-04", "2026-03-08", "2026-03-12"]
        ),
        "finish": pd.to_datetime(
            ["2026-03-04", "2026-03-09", "2026-03-13", "2026-03-15"]
        ),
        "owner": ["数据", "分析", "分析", "负责人"],
    }
)
diamonds = pd.read_csv("/datasets/diamonds.csv")
flights = pd.read_csv("/datasets/flights.csv")
gapminder = pd.read_csv("/datasets/gapminder.csv")
print(f"Diamonds {len(diamonds):,} | Flights {len(flights):,} | Gapminder {len(gapminder):,} 行")


In [ ]:
# 2️⃣ 特征工程：为各章图表构造分析所需的派生字段
orders_full = diamonds.assign(
    date=pd.Timestamp("2026-01-01"),
    category=diamonds["cut"],
    region=diamonds["clarity"],
    channel=diamonds["color"],
    order_value=diamonds["price"],
    items=diamonds["carat"],
    sales=diamonds["price"],
    month="公开样本",
)
orders = orders_full.sample(5_000, random_state=55)

monthly = (
    flights.query("year == 1960")
    .rename(columns={"passengers": "sales"})
    .copy()
)
monthly["orders"] = monthly["sales"]
monthly["profit"] = monthly["sales"].rolling(3, min_periods=1).mean()

regional = orders_full.groupby(["region", "channel"], as_index=False)[
    "sales"
].sum()

hierarchy = (
    diamonds.groupby(["cut", "color"], as_index=False)["price"]
    .sum()
    .rename(
        columns={"cut": "department", "color": "category", "price": "sales"}
    )
)

countries = gapminder.query("year == 2007").assign(
    country=lambda frame: frame["country"],
    market=lambda frame: frame["country"],
    sales=lambda frame: frame["gdpPercap"],
    growth=lambda frame: frame["lifeExp"],
)
print(f"样本：orders {len(orders):,} | monthly {len(monthly):,} 行")


## 77.6 基础图表

先保留必要的编码：位置、颜色或大小。图表标题、坐标轴和单位应能让读者脱离代码理解结果。


In [ ]:
fig = px.funnel(funnel, x="users", y="stage", title="用户转化漏斗")
fig.update_traces(textinfo="value+percent initial+percent previous")
fig.update_layout(xaxis_title="用户数", yaxis_title="阶段")
fig.show()


**练一练**：上面基础漏斗展示了各阶段人数和整体转化信息，现在请动手只改一个图表参数或数据字段，观察图形会怎么变。比如把 `textinfo` 从 `"value+percent initial+percent previous"` 改成 `"value+percent previous"`，让它只显示「相对上一个阶段」的转化率；或者把「加入购物车」阶段的用户数调小一些，模拟该环节流失更严重，看漏斗在这一段会不会明显收窄。运行并核对自检打印，然后回答：你改了什么、图上多了或少了什么信息、这一步更强调整体转化还是相邻环节的流失。


In [ ]:
# 请在下方填写代码：完成填空，复现基础漏斗并只微调一个参数或数据字段。
# _A_：只保留「相邻阶段转化率」的 textinfo 取值（示例为 "value+percent previous"）
# _B_：「加入购物车」阶段新的用户数（示例 2300，比原来 3100 更小，模拟更多流失）


In [ ]:
# 讲解：textinfo 换成 "value+percent previous" 后，标签只显示「相对上一个阶段」的转化率，
# 便于看清相邻环节的流失强度；把「加入购物车」人数从 3100 调小到 2300，这段漏斗会明显收窄，
# 一眼就能看出它是拉低整体转化率的关键环节。
pedido_textinfo = "value+percent previous"
funnel3 = funnel.copy()
funnel3.loc[funnel3["stage"] == "加入购物车", "users"] = 2300

fig = px.funnel(funnel3, x="users", y="stage", title="用户转化漏斗（相邻阶段版）")
fig.update_traces(textinfo=pedido_textinfo)
fig.show()


## 77.7 进阶变体

在基础图表可读的前提下增加分组、布局、注释或交互。新增编码必须服务于一个明确问题。


In [ ]:
funnel_detail = pd.DataFrame(
    {
        "stage": funnel["stage"].tolist() * 2,
        "channel": ["自然流量"] * len(funnel) + ["广告"] * len(funnel),
        "users": [7000, 4500, 2100, 1280, 1010, 5000, 2700, 1000, 570, 410],
    }
)
fig = px.funnel(
    funnel_detail, x="users", y="stage", color="channel", title="分渠道转化漏斗"
)
fig.update_layout(xaxis_title="用户数", yaxis_title="阶段", legend_title="渠道")
fig.show()


## 77.8 参数说明

- orientation：方向
- textinfo：标签
- funnelmode：分组
- category_orders：阶段顺序


## 77.9 结果解读

计算阶段转化率和累计转化率，重点定位最大流失环节。


## 77.10 本章实训：交互图与信息层次

这一组实验专门训练“观察一个结果 → 只改一个变量 → 解释变化”。先运行第一个代码单元格，再运行第二个。


In [ ]:
import pandas as pd
import plotly.express as px

report = pd.DataFrame(
    {
        "region": ["华东", "华南", "华北", "西南"],
        "sales": [320, 250, 280, 190],
    }
)
fig = px.bar(report, x="region", y="sales", title="地区销售额")
fig.show()


### 77.10.1 第一个结果怎么读

Plotly 的基本流程是：准备表格、映射字段、设置标题、显示图形。悬停提示只能补充信息，不能替代坐标轴和单位。

请记录：输入是什么、输出是什么、输出支持了哪一个结论。


In [ ]:
fig = px.bar(
    report.sort_values("sales", ascending=False),
    x="region",
    y="sales",
    text="sales",
    title="按销售额排序的地区销售额",
)
fig.update_traces(textposition="outside")
fig.update_layout(yaxis_title="销售额", xaxis_title="地区")
fig.show()


### 77.10.2 第二个结果怎么读

第二个实验增加数值标签并排序。请检查：标签是否遮挡、标题是否准确、图形是否仍然能在窄屏阅读。

迁移任务：把一个输入值、一个字段或一个图表参数换成自己的例子，再用一句话解释变化。


## 77.11 错误恢复：空数据还能不能画图

真实数据和真实代码都会出问题。本节先观察问题，再用一个明确的检查或修复步骤恢复运行。


In [ ]:
import pandas as pd
import plotly.express as px

report = pd.DataFrame({"region": ["华东", "华南"], "sales": [120, 150]})
if report.empty:
    print("没有可绘制的数据，请先检查筛选条件。")
else:
    fig = px.bar(report, x="region", y="sales", title="地区销售额")
    fig.update_layout(yaxis_title="销售额", xaxis_title="地区")
    fig.show()


### 77.11.1 错误恢复步骤

1. 先看错误类型、字段或数据形状。
2. 判断问题发生在输入、处理中间结果还是输出。
3. 修复后重新检查结果，而不是只让代码不报错。

筛选后先判断是否为空，再调用绘图函数。空表不是绘图库的问题，而是上游筛选口径需要检查。

迁移任务：把示例中的输入换成一组会触发问题的数据，并记录你的修复规则。


## 77.12 易错点提醒

- 阶段不是同一批用户
- 忽略时间窗口
- 只看绝对流失不看转化率


## 77.13 练习与作业

请使用同一份数据完成下面任务，并说明你选择该图表的原因。完成后补充：图表回答了什么问题、最重要的视觉信号是什么、还有哪些信息无法从图中得出。


## 77.14 独立迁移练习

在默认图可读的前提下，增加一个 hover 字段或筛选交互。

先在下面单元格完成自己的版本；需要参考时再回看紧邻的示例或参考实现。


In [ ]:
# 独立迁移练习：换一个转化指标，观察不同阶段的流失差异
# 【目标】换 x 指标，练习从金额角度而非人数角度看转化。
import plotly.express as px

# 起点示例(已可运行)：把 users 换成金额，比较用户漏斗与金额漏斗的差异。
funnel_money = funnel.assign(money=funnel["users"] * 3.2)
fig = px.funnel(funnel_money, x="money", y="stage", title="用户转化金额漏斗")
fig.update_traces(textinfo="value+percent initial+percent previous")
fig.update_layout(xaxis_title="金额（元）", yaxis_title="阶段")
fig.show()

# ---- 反思记录：人数漏斗与金额漏斗，流失环节是否一致 ----
change_note = "待填写"
expected_change = "待填写"
observed_change = "运行后填写"
print(f"改动：{change_note}")
print(f"预期：{expected_change}")
print(f"观察：{observed_change}")


In [ ]:
practice_funnel = funnel.copy()
practice_funnel["previous_rate"] = practice_funnel["users"] / practice_funnel[
    "users"
].shift(1)
# 漏斗图不支持连续色阶，把转化率区间归为离散类别，再用 color 按类别着色


def _band(r):
    if pd.isna(r):
        return "首段"
    if r < 0.5:
        return "低"
    if r < 0.7:
        return "中"
    return "高"


practice_funnel["conversion_band"] = practice_funnel["previous_rate"].apply(
    _band
)
print(practice_funnel)
fig = px.funnel(
    practice_funnel,
    x="users",
    y="stage",
    color="conversion_band",
    title="转化率着色漏斗",
)
fig.update_layout(legend_title="相邻阶段转化率")
fig.show()


## 77.15 小结

用漏斗宽度展示流程各阶段人数和转化损失。


### 77.15.1 你已经掌握

- 判断漏斗图（px.funnel）的适用场景
- 准备与图表匹配的数据结构
- 从基础图表扩展到分组、注释或交互变体
- 按照业务问题解读图表并说明结论边界


### 77.15.2 关键参数

| 参数 | 作用 |
| --- | --- |
| `orientation` | 方向 |
| `textinfo` | 标签 |
| `funnelmode` | 分组 |
| `category_orders` | 阶段顺序 |


### 77.15.3 需要注意

- 阶段不是同一批用户
- 忽略时间窗口
- 只看绝对流失不看转化率


### 77.15.4 完成检查

- [ ] 能判断什么问题适合使用漏斗图（px.funnel）
- [ ] 能准备符合要求的数据结构
- [ ] 能独立完成基础图表和一个进阶变体
- [ ] 能调整关键参数并解释视觉变化
- [ ] 能根据图表写出有边界的数据结论


### 77.15.5 下一步推荐

把同一图表迁移到另一份数据，先保留同样的编码，再只改变一个维度。比较迁移前后的可读性，并说明哪些结论仍然成立。
